In [16]:
import splink.comparison_library as cl
from splink import DuckDBAPI, Linker, SettingsCreator, block_on, splink_datasets
import pandas as pd

db_api = DuckDBAPI()

df = pd.read_csv('new_data/new_csrankings.csv', header = None)
df.columns = ['clean_name', 'full_name', 'first_name', 'surname', 'affiliation', 'homepage', 'scholar', 'middle_name']
df['index'] = df.index


In [17]:
# Sanitize common text columns to remove newlines, unbalanced quotes/backslashes, and '\\N' placeholders
cols = ['clean_name','full_name','first_name','surname','affiliation','homepage','scholar','middle_name']
for c in cols:
    df[c] = df[c].fillna('').astype(str)
    # replace CR/LF with a single space, remove quotes/backslashes, trim whitespace
    df[c] = df[c].str.replace(r'[\r\n]+', ' ', regex=True)
    df[c] = df[c].str.replace(r'["\\]+', '', regex=True)
    df[c] = df[c].str.strip()
    # convert literal '\\N' (common null marker) to empty string
    df[c] = df[c].replace('\\N', '')

# Sanitize scholar IDs: remove embedded newlines and trim whitespace
# This prevents embedded newlines from being written into the output CSV
df['scholar'] = df['scholar'].astype(str).str.replace(r'[\r\n]+', '', regex=True).str.strip()

In [18]:
df.shape

(30400, 9)

In [19]:
comparison_middle_name = {
    "output_column_name": "middle_name",
    "comparison_levels": [
        {
            "sql_condition": """
                middle_name_l IS NULL OR middle_name_l = '' OR
                middle_name_r IS NULL OR middle_name_r = ''
            """,
            "is_null_level": True,
            "label_for_charts": "missing"
        },
        {
            "sql_condition": """
                substr(middle_name_l, 1, 1) = substr(middle_name_r, 1, 1)
            """,
            "m_probability": .75,
            "u_probability": 0.15,
            "label_for_charts": "initials match"
        },
        {
            "sql_condition": """
                jaro_winkler_similarity(middle_name_l, middle_name_r) >= 0.85
            """,
            "m_probability": 0.75,
            "u_probability": 0.15,
            "label_for_charts": "fuzzy match"
        },
    ],
}

            
            
settings = SettingsCreator(
    link_type="dedupe_only",
    comparisons=[
        comparison_middle_name,
        cl.JaroWinklerAtThresholds("first_name", [0.9, 0.7]),
        cl.JaroWinklerAtThresholds("surname", [0.9, 0.10]),
        ],
        blocking_rules_to_generate_predictions=[
            block_on("surname", "affiliation")
        ],
        unique_id_column_name="index"
    )


linker = Linker(df, settings, db_api)

linker.training.estimate_probability_two_random_records_match(
    [block_on("surname", "affiliation")],
    recall=0.7,
)

linker.training.estimate_u_using_random_sampling(max_pairs=1e6)

linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("surname", "affiliation")
)

#linker.training.estimate_parameters_using_expectation_maximisation(block_on("dob"))


pairwise_predictions = linker.inference.predict(threshold_match_weight=-10)

clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    pairwise_predictions, 0.95
)

df_clusters = clusters.as_pandas_dataframe()

Probability two random records match is estimated to be  4.3e-05.
This means that amongst all possible pairwise record comparisons, one in 23,269.45 are expected to match.  With 462,064,800 total possible comparisons, we expect a total of around 19,857.14 matching pairs
You are using the default value for `max_pairs`, which may be too small and thus lead to inaccurate estimates for your model's u-parameters. Consider increasing to 1e8 or 1e9, which will result in more accurate estimates, but with a longer run time.
----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - first_name (no m values are trained).
    - surname (no m values are trained).

----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l."surname" = r."surname") AND (l."affiliation" = r."affiliation")

Parameter estimates will be made for the following 

In [20]:
pairwise_predictions.as_pandas_dataframe().nlargest(n=10, columns='match_probability')

,match_weight,match_probability,index_l,index_r,middle_name_l,middle_name_r,gamma_middle_name,first_name_l,first_name_r,gamma_first_name,surname_l,surname_r,gamma_surname,affiliation_l,affiliation_r,match_key
214,13.800756,0.99993,5783,5790,(Jing),Jing,0,Dave,Dave,3,Tian,Tian,3,Purdue University,Purdue University,0
226,13.800756,0.99993,5466,5467,(Daphne),Daphne,0,Danfeng,Danfeng,3,Yao,Yao,3,Virginia Tech,Virginia Tech,0
480,13.800756,0.99993,25595,25597,(Ray),Ray,0,Sungsoo,Sungsoo,3,Hong,Hong,3,George Mason University,George Mason University,0
539,13.800756,0.99993,26563,26565,(Kenneth),Kenneth,0,Ting-Hao,Ting-Hao,3,Huang,Huang,3,Pennsylvania State University,Pennsylvania State University,0
595,13.800756,0.99993,6426,6430,M. Rasmussen,Rasmussen,0,Diane,Diane,3,Pennington,Pennington,3,University of Strathclyde,University of Strathclyde,0
772,13.800756,0.99993,12547,12568,(Selena),Selena,0,Jing,Jing,3,He,He,3,Old Dominion University,Old Dominion University,0
920,13.800756,0.99993,13465,13500,L. Ribeiro,Ribeiro,0,João,João,3,,,3,Universidade NOVA de Lisboa,Universidade NOVA de Lisboa,0
952,13.800756,0.99993,13690,13701,(Jim),Jim,0,Jun,Jun,3,Xu,Xu,3,Georgia Institute of Technology,Georgia Institute of Technology,0
972,13.800756,0.99993,13473,13497,M. Pereira,Pereira,0,João,João,3,,,3,Universidade de Lisboa,Universidade de Lisboa,0
982,13.800756,0.99993,13473,13496,M. Pereira,Pereira,0,João,João,3,,,3,Universidade de Lisboa,Universidade de Lisboa,0


In [21]:
sorted_df_clusters = df_clusters.groupby('cluster_id').head()

In [22]:
duplicate_value = df_clusters.duplicated(subset=['cluster_id'], keep = False)
duplicate_i = duplicate_value[duplicate_value == True].index.values

In [23]:
duplicate_i

array([   31,    34,    35, ..., 30170, 30263, 30264], shape=(2638,))

In [24]:
duplicate_table = df_clusters.iloc[duplicate_i].sort_values(by='cluster_id')
duplicate_table.head(n=30)

,cluster_id,clean_name,full_name,first_name,surname,affiliation,homepage,scholar,middle_name,index
31,31,A. P. Vinod,A. P. Vinod 0001,A.,Vinod,Nanyang Technological University,http://www.ntu.edu.sg/home/asvinod,NOSCHOLARPAGE,P.,31
34,31,A. Prasad Vinod,A. Prasad Vinod,A.,Vinod,Nanyang Technological University,http://www.ntu.edu.sg/home/asvinod,NOSCHOLARPAGE,Prasad,34
35,31,A. Prasad Vinod,A. Prasad Vinod 0001,A.,Vinod,Nanyang Technological University,http://www.ntu.edu.sg/home/asvinod,NOSCHOLARPAGE,Prasad,35
39,39,A. S. M. Hoque,A. S. M. Hoque,A.,Hoque,BUET,https://cse.buet.ac.bd/faculty_list/detail/asm...,3-Sb7tMAAAAJ,S. M.,39
40,39,A. S. M. Latiful Hoque,A. S. M. Latiful Hoque,A.,Hoque,BUET,https://cse.buet.ac.bd/faculty_list/detail/asm...,3-Sb7tMAAAAJ,S. M. Latiful,40
52,52,A. W. Roscoe,A. W. Roscoe 0001,A.,Roscoe,University of Oxford,http://www.cs.ox.ac.uk/bill.roscoe,miixtKcAAAAJ,W.,52
53,52,A. William Roscoe,A. William Roscoe,A.,Roscoe,University of Oxford,http://www.cs.ox.ac.uk/bill.roscoe,miixtKcAAAAJ,William,53
105,105,Abdeltawab M. A. Hendawi,Abdeltawab M. A. Hendawi,Abdeltawab,Hendawi,University of Rhode Island,https://homepage.cs.uri.edu/~ahendawi,ad3Gki4AAAAJ,M. A.,105
106,105,Abdeltawab M. Hendawi,Abdeltawab M. Hendawi,Abdeltawab,Hendawi,University of Rhode Island,https://homepage.cs.uri.edu/~ahendawi,ad3Gki4AAAAJ,M.,106
120,119,Abdulla Khalid Al-Ali,Abdulla Khalid Al-Ali,Abdulla,Al-Ali,Qatar University,http://www.qu.edu.qa/engineering/computer/facu...,s1LawbAAAAAJ,Khalid,120


In [25]:
clean_table = df_clusters.drop_duplicates(subset=['cluster_id'], keep='first')
clean_table.drop(columns=['cluster_id', 'index'], inplace=True)


/var/folders/9d/n9kd385x17q1dpk5vtyw86d40000gn/T/ipykernel_99261/2865588344.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_table.drop(columns=['cluster_id', 'index'], inplace=True)


In [26]:
clean_table.head()

,clean_name,full_name,first_name,surname,affiliation,homepage,scholar,middle_name
0,A Min Tjoa,A Min Tjoa,A,Tjoa,TU Wien,http://www.ifs.tuwien.ac.at/tjoa,x8qCMhcAAAAJ,Min
1,A. Akbari Azirani,A. Akbari Azirani,A.,Azirani,IUST,http://ce.iust.ac.ir/page.php?slct_pg_id=6537&...,pCil4_cAAAAJ,Akbari
2,A. Akbariazirani,A. Akbariazirani,A.,Akbariazirani,IUST,http://ce.iust.ac.ir/page.php?slct_pg_id=6537&...,pCil4_cAAAAJ,N
3,A. Aldo Faisal,A. Aldo Faisal,A.,Faisal,Imperial College London,https://www.imperial.ac.uk/people/a.faisal,WjHjbrwAAAAJ,Aldo
4,A. Antony Franklin,A. Antony Franklin,A.,Franklin,IIT Hyderabad,http://www.iith.ac.in/~antony/index.html,LVfqLuoAAAAJ,Antony


In [27]:
# Final sanitization for the cleaned table before writing to CSV
cols = ['clean_name','full_name','first_name','surname','affiliation','homepage','scholar','middle_name']
for c in cols:
    clean_table[c] = clean_table[c].fillna('').astype(str)
    clean_table[c] = clean_table[c].str.replace(r'[\r\n]+', ' ', regex=True)
    clean_table[c] = clean_table[c].str.replace(r'["\\]+', '', regex=True)
    clean_table[c] = clean_table[c].str.strip()
    clean_table[c] = clean_table[c].replace('\\N', '')

# Diagnostics: list any rows that still contain problematic characters
import re
pattern = re.compile(r'["\\\r\n]')
mask = clean_table[cols].apply(lambda col: col.str.contains(pattern)).any(axis=1)
if mask.any():
    print('Warning: rows remaining with problematic chars (first 20):', mask.sum())
    display(clean_table.loc[mask, cols].head(20))
else:
    print('Sanitization complete: clean_table looks good.')

/var/folders/9d/n9kd385x17q1dpk5vtyw86d40000gn/T/ipykernel_99261/4256916561.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_table[c] = clean_table[c].fillna('').astype(str)
/var/folders/9d/n9kd385x17q1dpk5vtyw86d40000gn/T/ipykernel_99261/4256916561.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_table[c] = clean_table[c].str.replace(r'[\r\n]+', ' ', regex=True)
/var/folders/9d/n9kd385x17q1dpk5vtyw86d40000gn/T/ipykernel_99261/4256916561.py:6: SettingWithCopyWarning: 
A value is trying to 

Sanitization complete: clean_table looks good.


In [28]:
clean_table.to_csv('clean_csrankings.csv', index = False, header = ['clean_name','full_name','first_name','surname','affiliation','homepage','scholar','middle_name'])


